# Lorenz Circuit Chaos, Theoretical Document

PHYS 2326 — University Physics II 

Group: 2024-10294, 2024-10324, 2024-10386, 2024-10471, 2024-10491
Reference: K. M. Cuomo and A. V. Oppenheim, *Circuit Implementation of Synchronized Chaos with Applications to Communications*, Physical Review Letters 71(1), 65 to 68, 1993.

 ## 1. Introduction

In 1963, the meteorologist Edward Lorenz published a three-variable model of atmospheric convection that would become one of the most celebrated examples of deterministic chaos. The system he derived, now universally known as the Lorenz equations, demonstrated that even simple nonlinear ordinary differential equations can produce trajectories that appear random, exhibit sensitive dependence on initial conditions, and resist long-term prediction. These properties, while initially seen as obstacles to forecasting, were later recognized as potential assets in an entirely different domain: secure communications.

The crucial insight, due to Pecora and Carroll in 1990, was that two identical chaotic systems can be made to *synchronize*, that is, to converge to the same trajectory, when one drives the other with a suitable signal. This discovery opened the possibility of using chaos as a carrier for information. If a transmitter produces a chaotic waveform, a synchronized receiver can reconstruct that waveform independently, and any small perturbation added to the transmitted signal (encoding a message) can be detected by comparing the received signal to the receiver's local reconstruction.

In 1993, Cuomo, Oppenheim, and Strogatz demonstrated this idea in practice by building an electronic circuit that implements the Lorenz equations and using it to transmit and recover both analog and digital signals through chaotic masking and modulation. Their paper, the primary reference for this project, remains a landmark in the field because it bridges the gap between abstract dynamical-systems theory and tangible circuit implementation.

This theory notebook develops the complete theoretical framework underlying the Cuomo system, progressing from fundamental physics to stochastic analysis. The development is organized as follows.

Section 2 traces the physical origins of the Lorenz equations, starting from the Navier–Stokes equations governing fluid convection in a thin horizontal layer heated from below (Rayleigh–Bénard convection). Through a spectral truncation originally due to Saltzman, the infinite-dimensional partial differential equations are reduced to three coupled ODEs whose variables represent the intensity of convective rolling motion, the horizontal temperature variation, and the deviation of the vertical temperature profile from linearity. The three control parameters, $\sigma$ (the Prandtl number), $r$ (the normalized Rayleigh number), and $b$ (a geometric factor), are given precise physical meaning.

Section 3 analyzes the Lorenz system as a dynamical system. The fixed points are computed, their linear stability is determined through eigenvalue analysis, and the conditions under which the system transitions from steady convection to chaotic motion are identified. The strange attractor is characterized through its geometry, its invariant measure, and its Lyapunov exponents, which quantify the rates of exponential divergence and convergence along different directions in state space.

Section 4 develops the theory of chaotic synchronization. The Pecora–Carroll decomposition splits the Lorenz system into a drive subsystem and a response subsystem, and a Lyapunov stability argument proves that the response converges to the drive from arbitrary initial conditions. The circuit-scaled equations used in the Cuomo implementation are presented, the error dynamics are derived, and a Lyapunov function is constructed whose negative-definite time derivative guarantees exponential synchronization. Two communication schemes, signal masking and binary modulation, are then developed on this synchronization foundation.

Section 5 extends the entire framework to the stochastic domain. Physical noise sources in the circuit and channel are catalogued, the Wiener process and Itô calculus are introduced, and the Lorenz SDE with additive noise is formulated. The Fokker–Planck equation governing the evolution of the probability density is derived, and a stochastic Lyapunov analysis yields an explicit bound on the mean-squared synchronization error as a function of noise intensity. The section concludes with a discussion of numerical methods for SDEs, Euler–Maruyama and Milstein, that bridge the theoretical treatment to the computational notebook.

Section 6 synthesizes the key results and discusses the broader significance of chaotic communication in the context of modern cryptographic and spread-spectrum methods.

Throughout, the emphasis is on building a self-contained theoretical narrative: every equation is derived rather than asserted, every parameter is given physical meaning, and every result is connected to the circuit implementation of Cuomo et al. The notation follows the Cuomo paper where possible, with the original Lorenz variables $(x, y, z)$ and the circuit-scaled variables $(u, v, w)$ clearly distinguished.

---

*The following sections (2–5) are the main body of the notebook.*

---

## Overview and Connection to Other Parts

This notebook is the physical foundation of the project. It traces the Lorenz system from
its origin in fluid dynamics down to the three coupled ODEs used by every subsequent part.

| Part | Builds on Part 1 by … |
|------|----------------------|
| **Part 2** | Analyzing the fixed points and stability of these very ODEs |
| **Part 3** | Splitting the system into drive/response subsystems |
| **Part 4** | Using the circuit-scaled form $(u, v, w)$ for hardware realization |
| **Part 5** | Adding stochastic perturbations to these same ODEs |

The roadmap of this notebook:

1. Rayleigh–Bénard convection, the physical setup.
2. Navier–Stokes simplification, how Saltzman and Lorenz reduced PDEs to ODEs.
3. The standard Lorenz equations, the canonical three-ODE system.
4. Physical meaning of $\sigma$, $r$, $b$, what each parameter represents.
5. Parameter values $\sigma=16$, $r=45.6$, $b=4$, the Cuomo regime (chaos).
6. Circuit-scaled form, the $(u, v, w)$ rescaling used in Parts 3 and 4.

---
## 1.1 Rayleigh–Bénard Convection

Consider a thin layer of viscous fluid heated from below and cooled from above.
When the temperature difference $\Delta T$ is small, heat conducts upward and the fluid is at
rest. As $\Delta T$ increases beyond a critical value, convective rolls form: warm fluid
rises, cool fluid descends, and circulating cells appear.

**Setup parameters:**

| Symbol | Quantity | Role |
|--------|----------|------|
| $H$ | layer thickness | length scale |
| $\Delta T$ | top–bottom temperature difference | drives convection |
| $\nu$ | kinematic viscosity | resists motion |
| $\kappa$ | thermal diffusivity | smooths temperature |
| $\alpha$ | thermal expansion coefficient | links density to temperature |
| $g$ | gravitational acceleration | drives buoyancy |

Two dimensionless numbers characterize the system:

$$\text{Prandtl number:}\quad \sigma = \frac{\nu}{\kappa}
\qquad\qquad
\text{Rayleigh number:}\quad \text{Ra} = \frac{g\,\alpha\,\Delta T\,H^{3}}{\nu\,\kappa}$$

Convection begins at a critical Rayleigh number $\text{Ra}_c \approx 657.5$ (free-slip, conducting boundaries).
Above this threshold, the flow becomes increasingly complex as $\text{Ra}$ grows.

---
## 1.2 Governing PDEs (Boussinesq Approximation)

Under the Boussinesq approximation (density variations matter only in the buoyancy term),
the fluid motion is described by three coupled partial differential equations:

**Continuity (incompressibility):**
$$\nabla \cdot \mathbf{u} = 0$$

**Navier–Stokes momentum equation:**
$$\frac{\partial \mathbf{u}}{\partial t} + (\mathbf{u}\cdot\nabla)\mathbf{u}
= -\frac{1}{\rho_0}\nabla p + \nu \nabla^2 \mathbf{u} + g\,\alpha\,T\,\hat{\mathbf{z}}$$

**Heat (advection–diffusion):**
$$\frac{\partial T}{\partial t} + (\mathbf{u}\cdot\nabla)T = \kappa \nabla^2 T$$

Here $\mathbf{u}=(u_x, u_z)$ is the 2-D velocity field, $T$ is the deviation from a linear
conducting profile, $p$ is the pressure, and $\hat{\mathbf{z}}$ is the upward unit vector.

---
## 1.3 Lorenz's Galerkin Truncation

Saltzman (1962) and Lorenz (1963) sought a low-dimensional approximation by expanding
the velocity stream function $\psi$ and temperature $T$ in Fourier series and keeping only
the lowest-order terms:

$$\psi(x, z, t) = \frac{\sqrt{2}\,(1+a^2)\,\kappa}{a}\,X(t)\,\sin\!\left(\frac{\pi a x}{H}\right)\sin\!\left(\frac{\pi z}{H}\right)$$

$$\theta(x, z, t) = \frac{\sqrt{2}\,\Delta T}{\pi\,R_a/R_c}\!\left[Y(t)\cos\!\left(\frac{\pi a x}{H}\right)\sin\!\left(\frac{\pi z}{H}\right) - Z(t)\sin\!\left(\frac{2\pi z}{H}\right)\right]$$

where $a$ is the wavenumber and $X(t), Y(t), Z(t)$ are time-dependent mode amplitudes.

Substituting these expansions into the PDEs, projecting onto each Fourier mode, and rescaling
time by the diffusive time-scale $\kappa/H^{2}$ produces the celebrated Lorenz equations:

$$\boxed{\begin{aligned}
\dot{x} &= \sigma(y - x) \\
\dot{y} &= rx - y - xz \\
\dot{z} &= xy - bz
\end{aligned}}$$

These three coupled, autonomous, nonlinear ODEs are the foundation for everything that follows
in the project.

---
## 1.4 Physical Meaning of $x$, $y$, $z$

Each variable represents the amplitude of a Fourier mode, not a physical position:

| Variable | Represents | Sign convention |
|----------|-----------|-----------------|
| $x(t)$ | Intensity of the convective overturning circulation | $x>0$, clockwise, $x<0$, counter-clockwise |
| $y(t)$ | Horizontal temperature variation between rising and falling currents | $y>0$, warm rising, cold falling |
| $z(t)$ | Distortion of the vertical temperature profile from linear | $z>0$, top-heavy distortion |

## 1.5 Physical Meaning of $\sigma$, $r$, $b$

| Parameter | Definition | Physical meaning |
|-----------|-----------|------------------|
| $\sigma = \dfrac{\nu}{\kappa}$ | Prandtl number | Ratio of momentum to thermal diffusivity. Large $\sigma$ → fluid "feels" viscous before it conducts heat. |
| $r = \dfrac{\text{Ra}}{\text{Ra}_c}$ | Reduced Rayleigh number | Dimensionless thermal driving. $r > 1$ ⟹ convection onsets. $r > r^* \approx 24.74$ ⟹ chaos can appear. |
| $b = \dfrac{4}{1+a^2}$ | Geometric factor | Determined by the aspect ratio of the convection rolls. |

## 1.6 Project Parameter Values

This project uses the Cuomo–Oppenheim values, chosen so that the resulting circuit signal
is broadband enough to mask information:

$$\boxed{\sigma = 16, \qquad r = 45.6, \qquad b = 4}$$

These differ from the textbook Lorenz values $(\sigma=10,\, r=28,\, b=8/3)$ but produce
qualitatively similar chaotic behavior. Part 2 will confirm chaos at these specific values
via Lyapunov exponent analysis.

---
## 1.7 Circuit-Scaled Form (Bridge to Parts 3 & 4)

For implementation in op-amp circuits, the standard Lorenz variables (which range over
$\pm 20$ for $x, y$ and $0$–$50$ for $z$) must be rescaled to fit within the $\pm 10$ V op-amp range.
Cuomo introduces the substitution:

$$x = 10\,u, \qquad y = 10\,v, \qquad z = 20\,w$$

Plugging into the standard Lorenz equations and dividing through by the scaling factor
yields the circuit-scaled system used in Parts 3 and 4:

$$\boxed{\begin{aligned}
\dot{u} &= \sigma(v - u) \\
\dot{v} &= ru - v - 20\,uw \\
\dot{w} &= 5\,uv - bw
\end{aligned}}$$

The coefficients 20 and 5 in the circuit form arise directly from the scaling factors:
$\beta = 20$ (in $\dot{v}$) and $\alpha^{2}/\beta = 100/20 = 5$ (in $\dot{w}$).
Part 4 §4.3 derives this in detail.

---
## 1.8 Summary

Starting from the Navier–Stokes equations describing Rayleigh–Bénard convection, a Galerkin
truncation reduces the infinite-dimensional fluid dynamics to three coupled ordinary
differential equations: the Lorenz system. The state variables $(x, y, z)$ are mode amplitudes,
and the parameters $(\sigma, r, b)$ are direct dimensionless quantities of the convection setup.

At the project values $\sigma=16,\, r=45.6,\, b=4$, the system is in the chaotic regime,
as Part 2 will confirm. The circuit-scaled rescaling $x = 10u$, $y = 10v$, $z = 20w$ produces
the equivalent system used in Parts 3 and 4 for op-amp implementation.

**Forward to Part 2:** With the equations established, the next question is *why* this system
is chaotic, answered by analyzing the fixed points, their stability, and the Lyapunov exponents.

## Part 2 - Fixed Points, Stability & Chaos

In this notebook I look at the equilibrium points of the Lorenz system, check when they go unstable, and then see how the chaos shows up. After that I look at sensitive dependence on initial conditions, estimate the largest Lyapunov exponent, and plot the strange attractor for sigma = 16, r = 45.6, b = 4.

1. The Lorenz Equations

Same system from the first notebook:

$$\dot{x} = \sigma (y - x)$$
$$\dot{y} = r x - y - x z$$
$$\dot{z} = x y - b z$$

sigma is the Prandtl number, r is basically how hard the fluid is being driven (Rayleigh number), and b comes from the geometry of the convection rolls. All positive.

Even though the equations look pretty simple (only two nonlinear terms, xz and xy), the behavior changes a lot as r increases. Small r = everything dies out. Medium r = steady convection. Big enough r = chaos. The point of this notebook is to figure out where exactly that switch happens.

2. Equilibrium Points

Equilibria are where nothing is changing, so $\dot{x} = \dot{y} = \dot{z} = 0$. Setting the right hand sides to zero:

$$\sigma(y - x) = 0 \;\;\Rightarrow\;\; y = x$$
$$r x - y - x z = 0$$
$$x y - b z = 0$$

From the first equation y = x. Plugging into the third gives $z = x^2 / b$. Then the second equation becomes

$$x \left[(r - 1) - \frac{x^2}{b}\right] = 0.$$

So either x = 0 (giving the origin), or $x^2 = b(r-1)$. The second one only has real solutions when r > 1. So the three equilibria are:

- The origin $\mathbf{O} = (0, 0, 0)$, which is always there. Physically this is the "no convection" state.
- A pair $\mathbf{C^\pm} = (\pm \sqrt{b(r-1)}, \pm \sqrt{b(r-1)}, r - 1)$ that only exists once r > 1. These are the two convection rolls spinning in opposite directions.

So at r = 1 the origin "splits" into the two new equilibria - this is a pitchfork bifurcation.

Plugging in the numbers, $\sqrt{b(r-1)} = \sqrt{4 \cdot 44.6} \approx 13.36$, so $C^\pm \approx (\pm 13.36, \pm 13.36, 44.6)$. These end up being the centers of the two "wings" of the butterfly later on.

3. Linearizing - the Jacobian

To check stability we look at what happens to small perturbations near each fixed point. If we write $\mathbf{x}(t) = \mathbf{x}^* + \delta \mathbf{x}(t)$ and only keep linear terms, we get $\dot{\delta \mathbf{x}} = J(\mathbf{x}^*) \delta \mathbf{x}$, where J is the Jacobian:

$$J(x, y, z) = \begin{pmatrix} -\sigma & \sigma & 0 \\ r - z & -1 & -x \\ y & x & -b \end{pmatrix}.$$

The eigenvalues of J at the fixed point tell us the story:

- All eigenvalues have negative real part -> stable, perturbations shrink.
- Any eigenvalue has positive real part -> unstable.
- Complex eigenvalues mean spiraling (in or out depending on the sign of the real part).

Looking at the output: the origin has one positive eigenvalue and two negative ones, so it's a saddle - trajectories get pushed away in one direction and pulled in along the other two.

For C+ and C- there's one negative real eigenvalue and a pair of complex conjugate eigenvalues with a *positive* real part. So they're unstable spirals. Basically a trajectory near C+ spirals outward, eventually gets thrown across to the other side near C-, spirals out again, and so on. That back-and-forth is what gives the butterfly its two lobes.

4. The Origin in More Detail

At the origin x = y = z = 0 so the Jacobian collapses to:

$$J(\mathbf{O}) = \begin{pmatrix} -\sigma & \sigma & 0 \\ r & -1 & 0 \\ 0 & 0 & -b \end{pmatrix}.$$

The bottom row decouples and immediately gives an eigenvalue $-b < 0$. The remaining 2x2 block has characteristic equation $\lambda^2 + (\sigma + 1)\lambda - \sigma(r - 1) = 0$, with roots:

$$\lambda_{1,2} = \frac{-(\sigma + 1) \pm \sqrt{(\sigma + 1)^2 + 4\sigma(r - 1)}}{2}.$$

For r < 1 both of these are negative, so the origin is stable - any disturbance just dies out (pure heat conduction). Right at r = 1 one of them hits zero, and for r > 1 it goes positive and the origin becomes a saddle. This is the first bifurcation in the Lorenz system: the fluid prefers to start convecting instead of just sitting there conducting heat.

You can see the largest real eigenvalue crossing zero exactly at r = 1, just like the algebra said. Below that the origin attracts, above it the origin pushes things away.

5. C+ and C- - the Hopf Bifurcation

The non-trivial fixed points are stable for r just slightly above 1 (this is the steady convection regime). But as r grows, the complex eigenvalues of $J(C^\pm)$ drift across the imaginary axis. The exact value of r where their real part hits zero is

$$r_H = \sigma \, \frac{\sigma + b + 3}{\sigma - b - 1}.$$

This is the Hopf bifurcation. Once r > $r_H$, both C+ and C- are unstable, and at that point there's nowhere stable for trajectories to settle - they have to keep moving forever inside some bounded region. That's where the strange attractor comes from.

For sigma = 16 and b = 4 the Hopf threshold works out to about 16 * 23 / 11 ~ 33.45. Our r = 45.6 is well past that, which is the main reason the system is chaotic at these parameters - neither convection roll is a place where things can settle down anymore.

6. So Why Do sigma = 16, r = 45.6, b = 4 Give Chaos?

Three things have to happen at the same time for the Lorenz attractor to show up:

1. Every fixed point has to be unstable. The origin is a saddle whenever r > 1. The Hopf threshold formula gives $r_H \approx 33.45$ for our sigma and b, and r = 45.6 clears that easily. So nothing stable left.

2. Trajectories have to stay in a bounded region. Otherwise they'd just shoot off to infinity and there'd be no attractor at all. There's a classical argument using the function $V = r x^2 + \sigma y^2 + \sigma (z - 2r)^2$ - on a big enough ellipsoid V always decreases, so trajectories can't escape.

3. The flow has to contract volumes. The divergence of the vector field is $-\sigma - 1 - b$, which is -21 for our parameters. So phase-space volumes shrink like $e^{-21 t}$. That means the long-term motion has to live on something with zero volume - but since the dynamics is also unstable in some directions, that "something" can't be just a point or a closed curve. It ends up being a fractal.

Putting it together: trajectories are trapped, kicked away from every fixed point, and squeezed onto a thin set. The only way all three can be true at the same time is a strange attractor with sensitive dependence on initial conditions. The values sigma = 16, r = 45.6, b = 4 land safely past $r_H$ but not so far out that they hit one of the periodic windows that show up at much larger r, so they reliably give chaos.

7. Sensitive Dependence on Initial Conditions

The classic test for chaos: take two trajectories that start ridiculously close and watch them pull apart. I'll start them $10^{-8}$ apart in x and integrate.

For a while the two trajectories track each other almost perfectly, then suddenly they pull apart. On the log plot the early part is basically a straight line - that slope is the largest Lyapunov exponent. After a while the gap stops growing because both trajectories are still stuck on the same bounded attractor, so they can only ever be a few tens of units apart at most.

This is what "sensitive dependence" really means in practice: even if you measure the initial state crazy precisely, after a finite amount of time you have no idea where the system actually is.

8. Largest Lyapunov Exponent

To get an actual number for the exponential growth rate, I'll use the Benettin algorithm: integrate two nearby trajectories, every so often rescale the separation back to the original tiny value $d_0$, and keep adding up the log of how much it grew. The largest Lyapunov exponent is then

$$\lambda_1 \approx \frac{1}{N \, \tau} \sum_{k=1}^{N} \ln \frac{d_k}{d_0},$$

where $d_k$ is the separation right before each rescaling and $\tau$ is the time between rescalings.

A positive lambda_1 is basically the formal proof of chaos. The reciprocal $1/\lambda_1$ is a rough "predictability horizon" - past a few of those time units, an initial uncertainty of $10^{-8}$ has blown up to order one and predictions are useless. For comparison, the textbook Lorenz parameters (sigma, r, b) = (10, 28, 8/3) give $\lambda_1 \approx 0.9$, so what we're getting here is in the same neighborhood.

9. The Strange Attractor

Last thing - actually plot the attractor. I'll integrate for a long time, throw away the beginning (since the trajectory has to spiral in onto the attractor first), and plot what's left.

And there's the butterfly. Each lobe wraps around one of the unstable spirals C+ and C-. The trajectory spirals outward from C+, eventually gets close enough to the saddle at the origin that it gets flung over to the other side, spirals out near C-, and the whole thing repeats. The number of loops on each side before switching is basically random - which is exactly the unpredictability that the positive Lyapunov exponent is measuring.

It's called "strange" because even though it lives in a region whose volume is shrinking to zero, it's not just a curve or a surface - it has fractal structure. That comes from the fact that the flow is stretching things out in some directions (chaos) and folding them back into the bounded region at the same time.

10. Wrapping Up

So putting everything together:

- Three equilibria: the origin (always there), and the symmetric pair C+ and C- which appear at r = 1 in a pitchfork bifurcation.
- The origin is stable for r < 1 and a saddle for r > 1. The pair C+/C- are stable for a while, but go unstable at the Hopf threshold $r_H = \sigma(\sigma + b + 3)/(\sigma - b - 1)$.
- For sigma = 16 and b = 4 we get $r_H \approx 33.45$, so r = 45.6 has all three fixed points unstable. Combined with the trapping region and volume contraction (divergence = -21), this forces a strange attractor.
- The two-trajectory experiment confirmed exponential separation, and the Benettin estimate gave a positive largest Lyapunov exponent. That's what "chaos" actually means in the technical sense.
- The strange attractor itself is the butterfly: trajectories spiraling out from each unstable spiral and switching lobes irregularly forever.

# Part 3, Synchronization Theory
## `03-synchronization.ipynb`

**Course:** Digital and Multimedia Watermarking, Spring 2026  
**Group:** 2024-10294 · 2024-10324 · 2024-10386  
**Reference:** K. M. Cuomo & A. V. Oppenheim, *Circuit Implementation of Synchronized Chaos
with Applications to Communications*, Physical Review Letters 71(1), 65–68, 1993.

---

## Overview

This notebook develops the mathematical core of the project: proving that two Lorenz
circuits can synchronize, and that this synchronization is guaranteed to happen exponentially
fast regardless of starting conditions.

| Builds on | Leads into |
|-----------|------------|
| **Part 1:** Circuit-scaled Lorenz ODEs, parameters $\sigma$, $r$, $b$ | **Part 4:** Synchronization enables message recovery in both communication schemes |
| **Part 2:** Chaos confirmed at $\sigma=16$, $r=45.6$, $b=4$ | **Part 5:** Noise perturbs the synchronized state studied here |

Three things are proved:

1. Pecora–Carroll decomposition, how to split the Lorenz system into a drive and a response subsystem.
2. Error dynamics, the equations governing how the difference between transmitter and receiver evolves.
3. Lyapunov stability proof, a rigorous argument that the error decays to zero exponentially fast, from any initial condition.

---
## 3.1 Pecora–Carroll Decomposition

Pecora and Carroll (1990) showed that certain subsystems of a chaotic system can be made to
synchronize when driven by a common signal from the full system. The key is to find a
drive variable whose signal, injected into an identical copy of the remaining subsystem,
forces that copy to track the original.

### The circuit-scaled Lorenz transmitter (from Part 1)

$$\begin{aligned}
\dot{u} &= \sigma(v - u) \\
\dot{v} &= ru - v - 20\,u\,w \\
\dot{w} &= 5\,u\,v - b\,w
\end{aligned}
\qquad \sigma = 16,\; r = 45.6,\; b = 4$$

We designate $u(t)$ as the drive variable. The transmitter broadcasts $u(t)$ to the
receiver over the channel.

### Receiver equations, response subsystem driven by $u(t)$

The receiver replaces $u$ in the $(v, w)$ equations with the incoming transmitter signal,
and adds an autonomous $u_r$ equation:

$$\begin{aligned}
\dot{u}_r &= \sigma(v_r - u_r) & &\text{(autonomous, converges once } v_r \to v\text{)} \\
\dot{v}_r &= r\,u - v_r - 20\,u\,w_r & &\text{(driven by transmitter } u\text{)} \\
\dot{w}_r &= 5\,u\,v_r - b\,w_r & &\text{(driven by transmitter } u\text{)}
\end{aligned}$$

The receiver state $(u_r, v_r, w_r)$ may start from completely different initial
conditions than the transmitter. The synchronization theorem will show it converges regardless.

---
## 3.2 Error Dynamics

Define the synchronization error:

$$e_1 = u - u_r, \qquad e_2 = v - v_r, \qquad e_3 = w - w_r$$

Subtract receiver equations from transmitter equations term by term.

**Error in $u$:**
$$\dot{e}_1 = \sigma(v - u) - \sigma(v_r - u_r) = \sigma(e_2 - e_1)$$

**Error in $v$:**
$$\dot{e}_2 = (ru - v - 20uw) - (ru - v_r - 20u\,w_r) = -e_2 - 20u(t)\,e_3$$

**Error in $w$:**
$$\dot{e}_3 = (5uv - bw) - (5u\,v_r - b\,w_r) = 5u(t)\,e_2 - b\,e_3$$

The complete error system is:

$$\boxed{\begin{aligned}
\dot{e}_1 &= \sigma(e_2 - e_1) \\
\dot{e}_2 &= -e_2 - 20\,u(t)\,e_3 \\
\dot{e}_3 &= 5\,u(t)\,e_2 - b\,e_3
\end{aligned}}$$

This is a non-autonomous linear system, the coefficient $u(t)$ is the chaotic transmitter
output. The $(e_2, e_3)$ subsystem evolves independently of $e_1$.
Once $e_2, e_3 \to 0$, the $\dot{e}_1$ equation gives $\dot{e}_1 = -\sigma e_1$,
forcing $e_1 \to 0$ exponentially with rate $\sigma = 16$.

---
## 3.3 Lyapunov Stability Proof

### Step 1, Lyapunov function

$$E(e,t) = \frac{1}{2}\left(\frac{e_1^2}{\sigma} + e_2^2 + 4e_3^2\right)$$

$E \geq 0$ always; $E = 0$ only when $e_1 = e_2 = e_3 = 0$.
The weights $1/\sigma$ and $4$ are chosen to eliminate the $u(t)$ cross-terms (shown next).

### Step 2, Time derivative $\dot{E}$

$$\dot{E} = \frac{e_1\dot{e}_1}{\sigma} + e_2\dot{e}_2 + 4e_3\dot{e}_3$$

Substituting the error equations:

$$\frac{e_1\dot{e}_1}{\sigma} = e_1 e_2 - e_1^2$$

$$e_2\dot{e}_2 = -e_2^2 - 20u(t)\,e_2 e_3$$

$$4e_3\dot{e}_3 = 20u(t)\,e_2 e_3 - 4b\,e_3^2$$

The $\pm 20u(t)\,e_2 e_3$ terms cancel exactly:

$$\dot{E} = -e_1^2 + e_1 e_2 - e_2^2 - 4b\,e_3^2$$

### Step 3, Show $\dot{E} < 0$

Complete the square:

$$-e_1^2 + e_1 e_2 - e_2^2 = -\left(e_1 - \tfrac{1}{2}e_2\right)^2 - \tfrac{3}{4}e_2^2$$

Therefore:

$$\boxed{\dot{E} = -\left(e_1 - \tfrac{1}{2}e_2\right)^2 - \tfrac{3}{4}e_2^2 - 4b\,e_3^2 \leq 0}$$

Every term is non-positive. $\dot{E} = 0$ only at $e = 0$. Therefore $E(t)$ strictly
decreases along every non-zero trajectory: the error converges to zero.

### Step 4, Exponential convergence rate

Bounding each term gives $\dot{E} \leq -2\lambda E$ with $\lambda = \min(1, \tfrac{3}{4}, 2b)$.
At $b = 4$: $\lambda = 0.75$, so:

$$E(t) \leq E(0)\,e^{-1.5t}
\qquad \Longrightarrow \qquad
\|e(t)\| \leq C\,e^{-0.75t} \xrightarrow{t\to\infty} 0$$

Synchronization is proved: exponential convergence from any initial condition.

---
## 3.4 Summary

| Property | Result |
|----------|--------|
| Drive variable | $u(t)$ from transmitter |
| Response variables | $(v_r, w_r)$ driven by $u$; $u_r$ autonomous |
| Lyapunov function | $E = \frac{1}{2}\!\left(\frac{e_1^2}{\sigma} + e_2^2 + 4e_3^2\right)$ |
| $\dot{E}$ | $-\left(e_1 - \frac{1}{2}e_2\right)^2 - \frac{3}{4}e_2^2 - 4b\,e_3^2 \leq 0$ |
| Convergence rate | $\|e(t)\| \leq C\,e^{-0.75t}$ |
| Holds for | Any initial conditions, any chaotic $u(t)$ |

**Why the $u(t)$ terms cancel:** The weights $1/\sigma$, $1$, $4$ in $E$ are chosen so that
$\pm 20u(t)\,e_2 e_3$ appears with opposite signs in $e_2\dot{e}_2$ and $4e_3\dot{e}_3$
and cancels exactly. The chaotic nature of $u(t)$ is therefore irrelevant to stability.

**Connection to Part 4:** This proof guarantees that $u_r(t) \to u(t)$, which is precisely
what allows the receiver to strip the chaotic carrier in signal masking, and why the
synchronization error is small when transmitter and receiver parameters match in CSK.

### Two Communication Schemes

**Scheme 1, Chaotic Signal Masking.** 
The transmitter adds a low-amplitude message $m(t)$ to the chaotic carrier $u(t)$ before
transmission. The synchronized receiver reconstructs $u_r(t) \approx u(t)$ and subtracts
to recover the message, $\hat{m}(t) = s(t) - u_r(t)$.

**Scheme 2, Binary Chaos Shift Keying (CSK).**
Bits are encoded by toggling parameter $b$ between $b_0 = 4$ (bit 0) and $b_1 = 4.4$ (bit 1).
The receiver, always using $b_0$, detects the bit through synchronization error power.

Both schemes exploit the broadband, noise-like appearance of the chaotic carrier to conceal
information from an eavesdropper who does not possess a matching receiver circuit.

---
## 4.1 Chaotic Signal Masking

### Principle

Signal masking (*additive chaotic hiding*) embeds a message $m(t)$ inside the chaotic carrier
$u(t)$ to produce the transmitted signal:

$$s(t) = u(t) + m(t)$$

To an eavesdropper, $s(t)$ is indistinguishable from the chaotic waveform alone, provided the
message amplitude is much smaller than the carrier:

$$|m(t)| \ll |u(t)|$$

The synchronized receiver (proved in **Part 3**) reconstructs $u_r(t) \approx u(t)$ and recovers
the message by subtraction:

$$\hat{m}(t) = s(t) - u_r(t) \approx u(t) + m(t) - u(t) = m(t)$$

### Receiver Equations

The receiver's response subsystem $(v_r, w_r)$ is driven by $s(t)$ in place of $u(t)$.
The autonomous $u_r$ equation converges once $v_r \to v$:

$$\begin{aligned}
\frac{du_r}{dt} &= \sigma(v_r - u_r) & \text{(autonomous; converges via } v_r \to v)\\
\frac{dv_r}{dt} &= r\,s(t) - v_r - 20\,s(t)\,w_r & \text{(driven by } s \text{)}\\
\frac{dw_r}{dt} &= 5\,s(t)\,v_r - b\,w_r & \text{(driven by } s \text{)}
\end{aligned}$$

### Signal-to-Masking Ratio (SMR)

Recovery quality is characterized by the signal-to-masking ratio:

$$\text{SMR} = \frac{\mathrm{rms}[u(t)]}{\mathrm{rms}[m(t)]}$$

High SMR (\u2265 20) gives excellent hiding and small recovery error. The error dynamics from Part 3
show that $m(t)$ enters the synchronization equations as a bounded perturbation of order
$\mathcal{O}(|m|/|u|)$, so the Lyapunov function $E = \tfrac{1}{2}(e_1^2/\sigma + e_2^2 + 4e_3^2)$
stays bounded and recovery error is proportional to $1/\text{SMR}$.

![Chaotic Signal Masking Block Diagram](figures/signal_masking_block.png)

**Figure 1.** Block diagram of the chaotic signal masking system. The transmitter adds the message
$m(t)$ to the chaotic carrier $u(t)$ producing $s(t) = u(t) + m(t)$. The receiver drives its
response subsystem $(v_r, w_r)$ with the incoming $s(t)$, reconstructs $u_r(t)$ via the autonomous
$u_r$ ODE, and recovers $\hat{m}(t) = s(t) - u_r(t)$. The dashed orange line shows $s(t)$
returning to the subtractor's positive input.

### Analysis: Signal Masking Results

The four-panel plot demonstrates the full masking and recovery pipeline.

**Row 1, Carrier $u(t)$:** The chaotic Lorenz attractor waveform with parameters confirmed
chaotic in Part 2 ($\sigma=16$, $r=45.6$, $b=4$). Its broadband, noise-like spectrum is what
makes the masking effective.

**Row 2, Message $m(t)$:** A pure sinusoid at 0.8 Hz with amplitude 0.05 V. This is the
information to be hidden and recovered. The amplitude is chosen so that $\text{SMR} \approx 11$,
which satisfies $|m| \ll |u|$.

**Row 3, Transmitted signal $s(t)$:** The sum $u(t) + m(t)$ looks indistinguishable from pure
chaos. An eavesdropper intercepting $s(t)$ sees only broadband noise and cannot extract $m(t)$
without the receiver circuit.

**Row 4, Recovered message $\hat{m}(t)$:** After synchronization converges (transient discarded
at $t < 20$ s), the recovered signal closely tracks the original. The residual RMSE is of order
$\mathcal{O}(1/\text{SMR})$, consistent with the error-dynamics prediction in Part 3.

> **Key insight:** Synchronization is the enabling technology. Without the Lyapunov stability
> proved in Part 3, $u_r(t)$ would not converge to $u(t)$ and recovery would fail entirely.

---
## 4.2 Binary Chaos Shift Keying (CSK)

### Principle

Binary CSK encodes information by modulating a system parameter rather than adding a separate
message signal. Each bit period of duration $T_b$ uses one of two transmitter configurations:

$$b(t) = \begin{cases} b_0 = 4.0 & \text{bit} = 0 \\ b_1 = 4.4 & \text{bit} = 1 \end{cases}$$

The transmitted signal is still the chaotic waveform $u(t)$, but its statistical character
changes subtly depending on the active parameter.

### Detection via Synchronization Error

The receiver always uses $b_0 = 4$ and is driven by the received $u(t)$:

$$\frac{du_r}{dt} = \sigma(v_r - u_r), \quad
\frac{dv_r}{dt} = r\,u - v_r - 20\,u\,w_r, \quad
\frac{dw_r}{dt} = 5\,u\,v_r - b_0\,w_r$$

The synchronization error $e(t) = u(t) - u_r(t)$ behaves differently for each bit:

- **Bit 0** ($b_{\text{TX}} = b_{\text{RX}} = 4$): parameters match → synchronization holds
  → small $|e(t)|$.
- **Bit 1** ($b_{\text{TX}} = 4.4 \neq b_{\text{RX}} = 4$): parameters mismatch → synchronization
  degrades → large $|e(t)|$.

### Bit Detection Rule

Integrate the squared error over each bit period to obtain the error power:

$$P_e^{(k)} = \frac{1}{T_b} \int_{kT_b}^{(k+1)T_b} e^2(t)\,dt, \qquad k = 0, 1, 2, \ldots$$

Then apply a threshold $\theta$ (chosen as the midpoint between the expected $P_e$ values for
each bit):

$$\hat{b}_k = \begin{cases} 0 & P_e^{(k)} < \theta \\ 1 & P_e^{(k)} \geq \theta \end{cases}$$

### Why $b_1 = 4.4$?

Part 2 showed that the Lorenz system remains chaotic for small variations of $b$ around 4.
The value $b_1 = 4.4$ is large enough to produce a measurably different error power but small
enough that the carrier still appears chaotic, preserving the security property.

![Binary CSK Block Diagram](figures/csk_block.png)

**Figure 2.** Binary Chaos Shift Keying (CSK) block diagram. The transmitter switches $b$ between
$b_0 = 4$ and $b_1 = 4.4$ according to the bit stream. The receiver always uses $b_0$. The
synchronization error power $P_e^{(k)}$ per bit period is compared to threshold $\theta$ to
produce the detected bit $\hat{b}$. The dotted blue line shows $u(t)$ providing the reference
for error computation.

### Analysis: Binary CSK Results

**Row 1, Bit sequence:** The test sequence `[0,1,0,0,1,1,0,1]` with $T_b = 5$ s per bit.
Each bit period is labeled with the active parameter $b$.

**Row 2, Chaotic carrier:** The carrier $u(t)$ visually appears the same regardless of whether
$b = 4$ or $b = 4.4$ is active. This is the security property of CSK, an eavesdropper cannot
determine the bit just by looking at $u(t)$.

**Row 3, Error power:** The synchronization error power $P_e^{(k)}$ is clearly larger during
bit-1 periods (parameter mismatch) than during bit-0 periods (parameters matched). The threshold
$\theta$ falls cleanly between the two power levels, enabling reliable detection.

**Row 4, Detection:** The detected sequence matches the transmitted sequence.
Residual errors, if any, arise because the synchronization has not fully settled at the start
of each bit period, a transient effect that limits bit rate. In practice, $T_b$ must be
several synchronization time-constants long.

> **Connection to Part 2:** The parameter values $b_0=4$ and $b_1=4.4$ were both shown to
> be chaotic in the bifurcation analysis of Part 2. If $b_1$ were large enough to escape the
> chaotic regime, the receiver would see a completely different attractor and the scheme
> would fail catastrophically.

---
## 4.3 Circuit Implementation

### Variable Rescaling: From $(x, y, z)$ to $(u, v, w)$

The standard Lorenz equations use dimensionless variables whose amplitude can reach $|x|, |y| \sim 20$
and $z \sim 50$. Op-amp circuits typically operate within $\pm 10$ V. Cuomo rescales:

$$x = \alpha \, u, \quad y = \alpha \, v, \quad z = \beta \, w$$

Substituting into the standard Lorenz equations:

$$\frac{du}{dt} = \sigma(v - u) \qquad \text{(unchanged)}$$

$$\frac{dv}{dt} = ru - v - \underbrace{\beta}_{\displaystyle 20}\, u\,w
\qquad \Rightarrow \quad \beta = 20$$

$$\frac{dw}{dt} = \underbrace{\frac{\alpha^2}{\beta}}_{\displaystyle 5}\, uv - bw
\qquad \Rightarrow \quad \alpha^2 = 5\beta = 100 \quad \Rightarrow \quad \alpha = 10$$

Therefore the circuit scaling is:

$$\boxed{x = 10\,u, \quad y = 10\,v, \quad z = 20\,w}$$

With the Lorenz attractor spanning roughly $x \in [-15, 15]$, this gives $u \in [-1.5, 1.5]$ V,
well within the $\pm 10$ V op-amp range. The complete circuit-scaled system is:

$$\begin{aligned}
\dot{u} &= \sigma(v - u) \\
\dot{v} &= ru - v - 20\,u\,w \\
\dot{w} &= 5\,u\,v - b\,w
\end{aligned}$$

with $\sigma = 16$, $r = 45.6$, $b = 4$ (or $b = 4.4$ for CSK bit 1).

### Op-Amp Circuit Realization

Each state variable is produced by an inverting integrator built from a resistor and capacitor:

$$V_{\text{out}}(t) = -\frac{1}{RC} \int V_{\text{in}}(\tau)\,d\tau$$

The nonlinear terms $u\,w$ and $u\,v$ are realized by analog multiplier ICs (Motorola/AD AD633
or equivalent). The AD633 outputs $XY/10$, so the circuit must account for this factor of 10
in the resistor network.

Time scaling. The Lorenz attractor has a characteristic time scale of roughly $\tau \sim 0.1$
(normalized). To shift the circuit bandwidth to audio/RF frequencies, all $RC$ products are
chosen to scale time. Cuomo uses $R = 10\,\text{k}\Omega$, $C = 10\,\text{nF}$, giving
$\tau_{RC} = RC = 100\,\mu\text{s}$, which maps the Lorenz second to $100\,\mu\text{s}$ of
circuit time (speedup factor $\approx 10^4$, placing the attractor bandwidth around 16 kHz).

**Component values for each coefficient:**

| Coefficient | Value | Implemented by | Resistor |
|-------------|-------|---------------|----------|
| $\sigma = 16$ | 16 | Resistor ratio at $u$-integrator input | $R_{\sigma} = R / 16 = 625\,\Omega$ |
| $r = 45.6$ | 45.6 | Resistor ratio at $v$-integrator input | $R_r = R / 45.6 \approx 219\,\Omega$ |
| $b = 4$ | 4 | Resistor ratio at $w$-integrator input | $R_b = R / 4 = 2.5\,\text{k}\Omega$ |
| Factor 20 (in $\dot{v}$) | 20 | AD633 gain + resistor ratio | $R_{20} = R / 20 = 500\,\Omega$ |
| Factor 5 (in $\dot{w}$) | 5 | AD633 gain + resistor ratio | $R_5 = R / 5 = 2\,\text{k}\Omega$ |

*Reference integrator:* $R = 10\,\text{k}\Omega$, $C = 10\,\text{nF}$ (all three integrators).

The AD633 multiplier has a built-in scaling of $1/10$, meaning its output is $XY/10$.
This reduces the effective coefficient by 10, which is compensated in the resistor ratios
above. For the $20 \cdot u\,w$ term in $\dot{v}$: the multiplier gives $u \cdot w / 10$,
so the amplifier stage must multiply by $200$ to get the net factor of $20$.

![Op-Amp Circuit Block Diagram](figures/circuit_diagram.png)

**Figure 3.** Block-level op-amp circuit diagram for the Cuomo Lorenz transmitter (Cuomo 1993,
Fig. 1). Each of the three state variables $(u, v, w)$ is generated by an inverting integrator
(blue, $R = 10\,\text{k}\Omega$, $C = 10\,\text{nF}$). Nonlinear coupling terms ($uw$ and $uv$)
are computed by AD633 analog multipliers (orange). Summing amplifiers (green) weight the input
terms by the resistor ratios before integration. The RC time constant $\tau = 100\,\mu\text{s}$
maps the Lorenz time unit to $100\,\mu\text{s}$ of circuit time, placing the attractor bandwidth
at approximately 16 kHz.

---
## Summary and Connection

### What Part 4 Established

This notebook demonstrated two working chaos-based communication schemes built on the Lorenz
system studied throughout the project:

| Scheme | Transmitter side | Receiver side | What is detected |
|--------|-----------------|---------------|------------------|
| Signal masking | $s(t) = u(t) + m(t)$ | Reconstruct $u_r \approx u$, subtract | Continuous waveform $m(t)$ |
| Binary CSK | Switch $b \in \{4, 4.4\}$ per bit | Fixed $b=4$ receiver | Discrete bits via error power $P_e^{(k)}$ |

Both depend on synchronization converging before the information changes. This establishes
a fundamental trade-off:

$$\underbrace{\text{bit rate}}_{{\displaystyle \propto 1/T_b}} \times
\underbrace{\text{synchronization time}}_{{\displaystyle \propto \lambda_{\max}^{-1}}} = \mathcal{O}(1)$$

where $\lambda_{\max}$ is the maximum Lyapunov exponent studied in Part 2.

###
The analysis in this notebook assumed a noiseless channel. In any real circuit or communication
system, thermal noise, quantization noise, and channel noise are unavoidable. Part 5 extends the
Lorenz system to a stochastic differential equation (SDE):

$$du = \sigma(v - u)\,dt + \sigma_n\,dW_1, \quad
dv = (ru - v - 20uw)\,dt + \sigma_n\,dW_2, \quad
dw = (5uv - bw)\,dt + \sigma_n\,dW_3$$

where $W_i$ are independent Wiener processes and $\sigma_n$ is the noise intensity. The
consequences for the two schemes in this notebook are:

- **Signal masking:** Noise raises the noise floor of $\hat{m}(t)$, reducing the effective SMR.
  Beyond a critical noise level the recovered message becomes unintelligible.
- **Binary CSK:** Noise spreads the distributions of $P_e^{(0)}$ and $P_e^{(1)}$ and increases
  the bit error rate. The threshold $\theta$ must be optimized as a function of $\sigma_n$.

Part 5 will quantify exactly how large $\sigma_n$ can be before each scheme fails, providing
a noise tolerance specification for the physical circuit.

# Part 5, Stochastic Extension of the Lorenz System

The preceding sections developed the Lorenz system as a purely deterministic dynamical system: given an initial condition $\mathbf{x}(0)$, the trajectory $\mathbf{x}(t)$ is uniquely determined for all future time by the ordinary differential equations

$$
\dot{\mathbf{x}} = \mathbf{f}(\mathbf{x}).
$$

In any physical realization, however, the system is inevitably subject to random perturbations. Electronic components exhibit thermal noise, analog multipliers introduce gain fluctuations, and communication channels corrupt transmitted signals. These disturbances are not merely a practical inconvenience; they fundamentally alter the mathematical character of the system, transforming a deterministic ODE into a *stochastic* differential equation (SDE). Understanding this transformation is essential for assessing whether chaotic synchronization, and, by extension, chaotic communication, remains viable under realistic conditions.

## 5.1 Physical Origins of Noise in Chaotic Circuits

Before introducing the mathematical formalism, it is worth cataloguing the physical mechanisms that inject randomness into a Lorenz circuit implementation. Each mechanism contributes noise with distinct statistical characteristics, but for the purposes of a first analysis they can all be aggregated into a single additive white-noise term.

### 5.1.1 Thermal (Johnson–Nyquist) Noise

Every resistor in the Lorenz circuit is a source of thermal noise. At thermal equilibrium, the random motion of charge carriers produces a fluctuating voltage across the resistor terminals. The power spectral density of this voltage is given by the Nyquist formula,

$$
S_V(f) = 4 k_B T R,
$$

where $k_B$ is Boltzmann's constant, $T$ is the absolute temperature, and $R$ is the resistance. This spectrum is flat (white) up to frequencies on the order of $k_B T / h \approx 6 \times 10^{12}$ Hz at room temperature, far above the bandwidth of any practical Lorenz circuit. Consequently, thermal noise is well modeled as additive white Gaussian noise (AWGN) within the circuit's operating band.

The Cuomo transmitter circuit employs over twenty resistors with values ranging from $1\,\text{k}\Omega$ to $200\,\text{k}\Omega$. Each contributes an independent noise source. When these are referred to the state-variable nodes $(u, v, w)$ through the operational-amplifier network, they produce effective noise voltages at each integrator output. Because the resistor values differ, the noise intensities at the three nodes are in general unequal, though for a simplified analysis one often adopts a single scalar noise intensity $\sigma_n$ as a representative magnitude.

### 5.1.2 Operational-Amplifier Noise

The operational amplifiers (LF353 in the Cuomo circuit) contribute two kinds of noise: an equivalent input voltage noise and an equivalent input current noise. The voltage noise is typically broadband with a spectral density on the order of $10$–$20\;\text{nV}/\sqrt{\text{Hz}}$ for a JFET-input amplifier, while the current noise is smaller but not negligible at high impedance nodes. Additionally, the AD632 analog multipliers used to implement the nonlinear terms $uv$ and $uw$ introduce their own broadband noise floor. Since these active-device noise sources are also spectrally flat within the circuit bandwidth, they reinforce the appropriateness of a white-noise model.

### 5.1.3 Channel Noise

When the Lorenz circuit is used for communication, the drive signal $u(t)$ must travel from the transmitter to the receiver through a physical channel, a wire, a radio link, or an optical fiber. Each of these channels adds noise. In the simplest model, the received signal is

$$
u_{\text{received}}(t) = u(t) + \nu(t),
$$

where $\nu(t)$ is a zero-mean white Gaussian process with spectral density $N_0/2$. This channel noise enters the receiver subsystem as an external perturbation to the drive input, and its effect on synchronization quality is one of the central questions motivating the stochastic analysis.

### 5.1.4 Component Tolerances and Parameter Mismatch

A subtler form of randomness arises from manufacturing tolerances. No two resistors, capacitors, or multipliers are identical. If the transmitter has parameters $(\sigma, r, b)$ and the receiver has slightly different values $(\sigma + \delta\sigma, r + \delta r, b + \delta b)$, perfect synchronization is impossible even in the absence of additive noise. Cuomo et al. investigated this numerically (their Fig. 13) and found that the Lorenz synchronizing system is reasonably robust to modeling errors up to about 10–15%, beyond which the output signal-to-noise ratio degrades significantly. While parameter mismatch is not the same as additive noise, it can be partially absorbed into the stochastic framework by treating the mismatch as a slowly varying random perturbation.

## 5.2 Mathematical Preliminaries: The Wiener Process and Itô Calculus

To place the noisy Lorenz system on rigorous mathematical footing, we require the language of stochastic calculus. The fundamental building block is the *Wiener process*, also known as standard Brownian motion.

### 5.2.1 Definition and Properties of the Wiener Process

A standard Wiener process $W(t)$, with $t \geq 0$, is a continuous-time stochastic process satisfying the following properties:

1. **Initial condition:** $W(0) = 0$ almost surely.

2. **Independent increments:** For any $0 \leq t_1 < t_2 \leq t_3 < t_4$, the increments $W(t_2) - W(t_1)$ and $W(t_4) - W(t_3)$ are independent random variables.

3. **Gaussian increments:** Each increment $W(t+\Delta t) - W(t)$ is normally distributed with mean zero and variance $\Delta t$:
$$
W(t + \Delta t) - W(t) \sim \mathcal{N}(0,\, \Delta t).
$$

4. **Continuous paths:** The sample paths $t \mapsto W(t)$ are continuous with probability one, but they are *nowhere differentiable*. This last property is crucial: one cannot write $dW/dt$ in the ordinary sense, which is precisely why stochastic calculus differs from classical calculus.

The variance of $W(t)$ itself grows linearly with time,

$$
\text{Var}[W(t)] = \mathbb{E}[W(t)^2] = t,
$$

which means the Wiener process diffuses away from the origin at a rate proportional to $\sqrt{t}$. In discrete approximation, over a time step $\Delta t$, the Wiener increment is

$$
\Delta W = W(t + \Delta t) - W(t) = \sqrt{\Delta t}\;\xi, \qquad \xi \sim \mathcal{N}(0, 1),
$$

a fact that will be important when we discuss numerical simulation methods.

### 5.2.2 The Itô Stochastic Differential Equation

A general Itô SDE for a state vector $\mathbf{x}(t) \in \mathbb{R}^n$ takes the form

$$
d\mathbf{x} = \mathbf{f}(\mathbf{x}, t)\,dt + \mathbf{G}(\mathbf{x}, t)\,d\mathbf{W}_t,
$$

where $\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^n$ is the *drift* vector (the deterministic part), $\mathbf{G}: \mathbb{R}^n \to \mathbb{R}^{n \times m}$ is the *diffusion* matrix (governing how noise enters each equation), and $\mathbf{W}_t = (W_1(t), \ldots, W_m(t))^\top$ is an $m$-dimensional vector of independent Wiener processes.

The notation $d\mathbf{x}$ is shorthand for the integral equation

$$
\mathbf{x}(t) = \mathbf{x}(0) + \int_0^t \mathbf{f}(\mathbf{x}(s), s)\,ds + \int_0^t \mathbf{G}(\mathbf{x}(s), s)\,d\mathbf{W}_s,
$$

where the second integral is an *Itô integral*. The Itô integral is defined as a limit of sums in which the integrand is evaluated at the *left endpoint* of each subinterval, in contrast to the Stratonovich convention which uses the midpoint. This choice is not merely a technicality; it determines the form of the chain rule.

### 5.2.3 Itô's Lemma

In ordinary calculus, if $x(t)$ satisfies an ODE and $V(x)$ is a smooth function, the chain rule gives $dV/dt = V'(x)\,\dot{x}$. In stochastic calculus the chain rule acquires an additional second-order correction. For a scalar SDE $dx = f\,dt + g\,dW$ and a twice-differentiable function $V(x, t)$, Itô's lemma states

$$
dV = \left(\frac{\partial V}{\partial t} + f\frac{\partial V}{\partial x} + \frac{1}{2}g^2 \frac{\partial^2 V}{\partial x^2}\right)dt + g\frac{\partial V}{\partial x}\,dW.
$$

The extra term $\tfrac{1}{2}g^2 V''$, absent from the deterministic chain rule, is a direct consequence of the fact that $(dW)^2 = dt$ to leading order. This lemma is the principal computational tool for analyzing functions of stochastic processes, and we will invoke it when studying how noise modifies Lyapunov functions for the synchronization error.

## 5.3 Stochastic Lorenz Equations

We are now in a position to write down the stochastic extension of the circuit-scaled Lorenz system. The simplest and most commonly adopted model adds independent white noise to each state equation with a uniform intensity $\sigma_n > 0$:

### 5.3.1 Additive Noise Model

$$
\boxed{
\begin{aligned}
du &= \sigma(v - u)\,dt + \sigma_n\,dW_1, \\
dv &= (ru - v - 20uw)\,dt + \sigma_n\,dW_2, \\
dw &= (5uv - bw)\,dt + \sigma_n\,dW_3,
\end{aligned}
}
$$

where $W_1, W_2, W_3$ are three mutually independent standard Wiener processes, and $\sigma_n$ is the noise intensity (in the same voltage units as $u$, $v$, $w$). This is an *additive* noise model because the diffusion coefficient $\sigma_n$ does not depend on the state $\mathbf{x}$. In the language of the general Itô SDE, the drift is the familiar Lorenz vector field $\mathbf{f}(\mathbf{x})$ and the diffusion matrix is $\mathbf{G} = \sigma_n \mathbf{I}_3$, the scalar $\sigma_n$ times the $3 \times 3$ identity.

The additive model is appropriate when the dominant noise sources are exogenous (thermal noise, channel noise) rather than state-dependent. For situations where noise enters multiplicatively, for instance, if the gain of an analog multiplier fluctuates in proportion to its output, one would instead write

$$
du = \sigma(v - u)\,dt + \sigma_n\,u\,dW_1,
$$

which is a *multiplicative* noise model. Multiplicative noise introduces qualitatively different behavior (noise-induced transitions, modified fixed-point locations), but for the Lorenz circuit the additive model is the standard first approximation and the one we adopt throughout.

### 5.3.2 Non-Uniform Noise Intensities

In a more refined model, each state variable may experience a different noise intensity, reflecting the different impedance environments in the circuit:

$$
\begin{aligned}
du &= \sigma(v - u)\,dt + \sigma_{n,1}\,dW_1, \\
dv &= (ru - v - 20uw)\,dt + \sigma_{n,2}\,dW_2, \\
dw &= (5uv - bw)\,dt + \sigma_{n,3}\,dW_3.
\end{aligned}
$$

Here the diffusion matrix becomes $\mathbf{G} = \text{diag}(\sigma_{n,1},\, \sigma_{n,2},\, \sigma_{n,3})$. One can estimate the individual $\sigma_{n,i}$ from the circuit topology by summing the noise contributions of all resistors referred to each integrator node. For the purposes of this theoretical development, however, we continue with the uniform case $\sigma_{n,1} = \sigma_{n,2} = \sigma_{n,3} = \sigma_n$ to keep expressions tractable.

## 5.4 Effects of Noise on Chaotic Dynamics

Adding noise to the Lorenz system has several profound consequences for the nature of the dynamics. These effects are not merely quantitative (trajectories get "fuzzy") but qualitative: the very framework for describing the system must change.

### 5.4.1 Trajectory Perturbation and Attractor Blurring

In the deterministic Lorenz system, the strange attractor is a fractal set of measure zero in $\mathbb{R}^3$: all trajectories eventually settle onto this thin, intricately folded structure. When additive noise is present, trajectories are continually kicked off the attractor by random impulses and then pulled back by the deterministic flow. The result is that the system no longer visits a sharp fractal set but instead occupies a *neighborhood* of the attractor whose width scales with $\sigma_n$.

Quantitatively, the instantaneous deviation $\delta \mathbf{x}$ induced by noise satisfies, to linear order,

$$
d(\delta \mathbf{x}) = \mathbf{J}(\mathbf{x}(t))\,\delta \mathbf{x}\,dt + \sigma_n\,d\mathbf{W}_t,
$$

where $\mathbf{J}(\mathbf{x})$ is the Jacobian of $\mathbf{f}$ evaluated along the (now stochastic) trajectory. Along directions where the Lorenz system is contracting (negative local Lyapunov exponent), perturbations are suppressed; along the expanding direction (positive Lyapunov exponent $\lambda_1 > 0$), perturbations are amplified. The net effect is that the attractor acquires a "thickness" that is largest along the unstable manifold and smallest along the most stable direction.

### 5.4.2 Loss of Deterministic Predictability

The deterministic Lorenz system is already unpredictable in a practical sense: small errors in initial conditions grow exponentially, with a characteristic doubling time of $1/\lambda_1$. Noise worsens this by continually injecting new uncertainty at every instant. Even if the initial condition were known exactly, the future trajectory would still be uncertain because the noise realization is unknown.

This means that in the stochastic setting, one must abandon the notion of predicting individual trajectories and instead work with *ensembles* of trajectories. The relevant objects become probability distributions $p(\mathbf{x}, t)$ rather than deterministic solutions $\mathbf{x}(t)$. The evolution of this probability density is governed by the Fokker–Planck equation, which we develop in Section 5.5.

### 5.4.3 Noise-Induced Effects on Invariant Measures

The deterministic Lorenz attractor supports a natural invariant measure $\mu_0$, informally, the long-time probability of finding the system in any given region of state space. When noise is added, this measure is replaced by a *stationary probability density* $p_s(\mathbf{x})$ that solves the stationary Fokker–Planck equation. For small $\sigma_n$, this density is concentrated near the deterministic attractor and approximately proportional to $\mu_0$ convolved with a Gaussian kernel of width $\sim \sigma_n$. For larger $\sigma_n$, the density spreads further, and can even "fill in" regions between the two lobes of the attractor that are rarely visited deterministically, qualitatively changing the statistics of the system.

## 5.5 The Fokker–Planck Equation

While the SDE describes the evolution of individual (stochastic) trajectories, the Fokker–Planck equation (FPE) describes the evolution of the probability density $p(\mathbf{x}, t)$ of the state vector. For the stochastic Lorenz system with additive noise, the FPE takes the form

$$
\frac{\partial p}{\partial t} = -\sum_{i=1}^{3} \frac{\partial}{\partial x_i}\bigl[f_i(\mathbf{x})\,p\bigr] + \frac{\sigma_n^2}{2}\sum_{i=1}^{3}\frac{\partial^2 p}{\partial x_i^2},
$$

or in compact vector notation,

$$
\frac{\partial p}{\partial t} = -\nabla \cdot (\mathbf{f}\,p) + \frac{\sigma_n^2}{2}\nabla^2 p.
$$

The first term on the right is the *advection* of probability by the deterministic flow: if there were no noise, $p$ would simply be transported along the Lorenz vector field, and any initial delta function $p(\mathbf{x}, 0) = \delta(\mathbf{x} - \mathbf{x}_0)$ would remain a delta function riding along the trajectory. The second term is *diffusion*: noise spreads probability out in all directions at a rate proportional to $\sigma_n^2/2$.

The Fokker–Planck equation is a *linear* partial differential equation for $p$, even though the underlying SDE is nonlinear in $\mathbf{x}$. This linearity is a significant theoretical advantage: while individual trajectories diverge chaotically, the probability density evolves smoothly and predictably. In principle, one can find the stationary density $p_s(\mathbf{x})$ by solving $\partial p / \partial t = 0$, although for the three-dimensional Lorenz system this generally requires numerical methods.

## 5.6 Noise and Synchronization: Stochastic Stability Analysis

The central question for chaotic communications is whether synchronization between the transmitter and receiver survives in the presence of noise. In the deterministic case (developed in Section 3), the synchronization error $\mathbf{e} = \mathbf{x}_{\text{transmitter}} - \mathbf{x}_{\text{receiver}}$ converges exponentially to zero, as demonstrated by the Lyapunov function

$$
E(\mathbf{e}) = \frac{1}{2}\left(\frac{e_1^2}{\sigma} + e_2^2 + 4e_3^2\right),
$$

whose time derivative was shown to be negative definite. In the stochastic case, perfect synchronization ($\mathbf{e} = 0$) is no longer achievable; instead, the error fluctuates around zero, and the relevant question becomes: how large are these fluctuations?

### 5.6.1 Stochastic Error Dynamics

Consider the case where additive noise enters both the transmitter and the channel. The transmitter evolves according to the stochastic Lorenz equations, and the receiver receives the noisy drive signal $u(t) + \nu(t)$. The error dynamics become

$$
\begin{aligned}
de_1 &= \sigma(e_2 - e_1)\,dt + \sigma_n\,dW_1^{(\text{ch})}, \\
de_2 &= (-e_2 - 20u\,e_3)\,dt + \sigma_n\,dW_2, \\
de_3 &= (5u\,e_2 - b\,e_3)\,dt + \sigma_n\,dW_3,
\end{aligned}
$$

where $dW_1^{(\text{ch})}$ represents channel noise entering through the drive signal, and $dW_2$, $dW_3$ represent noise within the receiver circuit itself. The crucial difference from the deterministic case is that even when $\mathbf{e}$ reaches the neighborhood of zero, the noise terms continually perturb it away.

### 5.6.2 Itô's Lemma Applied to the Lyapunov Function

Applying Itô's lemma to the Lyapunov function $E(\mathbf{e})$, we obtain

$$
dE = \underbrace{\left(\nabla E \cdot \mathbf{f}_e\right)}_{\text{deterministic (stabilizing)}}\,dt + \underbrace{\frac{\sigma_n^2}{2}\,\text{tr}\left(\mathbf{G}^\top \nabla^2 E\;\mathbf{G}\right)}_{\text{noise injection (destabilizing)}}\,dt + \text{(martingale term)}\,d\mathbf{W},
$$

where $\mathbf{f}_e$ is the drift of the error dynamics and $\nabla^2 E$ is the Hessian of $E$. The first $dt$ term is the same expression that was shown to be negative definite in the deterministic analysis, it represents the stabilizing tendency of synchronization. The second $dt$ term is new: it is always *positive* and represents the continuous injection of energy into the error by the noise.

For the Lyapunov function $E = \frac{1}{2}(e_1^2/\sigma + e_2^2 + 4e_3^2)$ with uniform noise $\mathbf{G} = \sigma_n \mathbf{I}$, the Hessian is $\nabla^2 E = \text{diag}(1/\sigma,\, 1,\, 4)$ and the noise-injection term evaluates to

$$
\frac{\sigma_n^2}{2}\left(\frac{1}{\sigma} + 1 + 4\right) = \frac{\sigma_n^2}{2}\left(\frac{1}{\sigma} + 5\right).
$$

With $\sigma = 16$, this gives approximately $\frac{\sigma_n^2}{2}(5.0625) \approx 2.53\,\sigma_n^2$.

### 5.6.3 Steady-State Error Bound

In the deterministic case, the time derivative $\dot{E}$ satisfies

$$
\dot{E} \leq -\gamma\, E,
$$

for some effective decay rate $\gamma > 0$ that depends on the Lorenz parameters. In the stochastic case, taking expectations and using the fact that the Itô integral has zero expectation, we get

$$
\frac{d}{dt}\mathbb{E}[E] \leq -\gamma\,\mathbb{E}[E] + \frac{\sigma_n^2}{2}\left(\frac{1}{\sigma} + 5\right).
$$

This is a first-order linear ODE for $\mathbb{E}[E]$. Its steady-state solution ($d\mathbb{E}[E]/dt = 0$) gives

$$
\boxed{
\mathbb{E}[E]_{\text{ss}} = \frac{\sigma_n^2}{2\gamma}\left(\frac{1}{\sigma} + 5\right).
}
$$

This is a key result. It says that the mean-squared synchronization error is proportional to $\sigma_n^2$ (the noise power) and inversely proportional to $\gamma$ (the synchronization strength). Synchronization is never perfect in the presence of noise, but the error can be made small if the noise is weak relative to the natural stability of the synchronization manifold.

The transient solution, starting from an initial error $\mathbb{E}[E](0) = E_0$, is

$$
\mathbb{E}[E](t) = E_0\,e^{-\gamma t} + \frac{\sigma_n^2}{2\gamma}\left(\frac{1}{\sigma} + 5\right)\left(1 - e^{-\gamma t}\right),
$$

which shows exponential convergence from $E_0$ to the noise floor $\mathbb{E}[E]_{\text{ss}}$.

### 5.6.4 Signal-to-Noise Ratio and Communication Quality

For the signal masking scheme, the recovered message is $\hat{m}(t) = s(t) - u_r(t) = m(t) + e_1(t)$, so the message recovery error is precisely the synchronization error $e_1$. The output signal-to-noise ratio is therefore

$$
\text{SNR}_{\text{out}} = \frac{\mathbb{E}[m^2]}{\mathbb{E}[e_1^2]} \approx \frac{P_m}{\sigma_n^2 / (\sigma\gamma)} = \frac{P_m \sigma \gamma}{\sigma_n^2},
$$

where $P_m = \mathbb{E}[m^2]$ is the message power. This expression makes the trade-offs explicit: higher synchronization strength $\gamma$ and higher $\sigma$ improve message recovery, while stronger noise degrades it quadratically.

Cuomo et al. found experimentally that for their circuit with $\sigma = 16$, $r = 45.6$, $b = 4$, the Lorenz synchronizing system and the extended Kalman filter produced comparable output SNRs over a wide range of input SNRs, with both exhibiting a threshold effect below which message recovery fails entirely. This threshold corresponds roughly to the regime where $\mathbb{E}[e_1^2]$ becomes comparable to the variance of the chaotic signal itself, at which point the receiver can no longer distinguish the synchronization error from the drive signal.

## 5.7 Numerical Methods for Stochastic Differential Equations

Standard deterministic integrators such as the fourth-order Runge–Kutta method (RK4) are not directly applicable to SDEs, because they do not account for the $O(\sqrt{dt})$ scaling of the noise increments. Specialized stochastic integrators are required, and the two most widely used are presented here.

### 5.7.1 The Euler–Maruyama Method

The Euler–Maruyama method is the stochastic analogue of the forward Euler method. For the SDE $d\mathbf{x} = \mathbf{f}(\mathbf{x})\,dt + \mathbf{G}\,d\mathbf{W}_t$, the discrete update is

$$
\mathbf{x}_{n+1} = \mathbf{x}_n + \mathbf{f}(\mathbf{x}_n)\,\Delta t + \mathbf{G}\,\Delta \mathbf{W}_n,
$$

where $\Delta \mathbf{W}_n = \sqrt{\Delta t}\;\boldsymbol{\xi}_n$ and each $\xi_{n,i} \sim \mathcal{N}(0,1)$ independently. This method has *strong order of convergence* $1/2$, meaning the expected error in any individual trajectory scales as $O(\sqrt{\Delta t})$, and *weak order of convergence* $1$, meaning errors in expected values of smooth functions of the solution scale as $O(\Delta t)$.

For the stochastic Lorenz system with $\sigma = 16$, $r = 45.6$, $b = 4$, and $\mathbf{G} = \sigma_n \mathbf{I}$, the explicit Euler–Maruyama update reads

$$
\begin{aligned}
u_{n+1} &= u_n + \sigma(v_n - u_n)\,\Delta t + \sigma_n \sqrt{\Delta t}\;\xi_{n,1}, \\
v_{n+1} &= v_n + (r\,u_n - v_n - 20\,u_n w_n)\,\Delta t + \sigma_n \sqrt{\Delta t}\;\xi_{n,2}, \\
w_{n+1} &= w_n + (5\,u_n v_n - b\,w_n)\,\Delta t + \sigma_n \sqrt{\Delta t}\;\xi_{n,3}.
\end{aligned}
$$

The time step $\Delta t$ must be chosen small enough to resolve both the deterministic dynamics (which for the Lorenz system with these parameter values have a characteristic timescale on the order of the inverse of the largest eigenvalue of the Jacobian, roughly $\sim 1/\sigma = 1/16$) and the noise fluctuations.

### 5.7.2 The Milstein Method

The Milstein method improves on Euler–Maruyama by including a correction term that accounts for the state-dependence of the diffusion coefficient. For a general scalar SDE $dx = f(x)\,dt + g(x)\,dW$, the Milstein update is

$$
x_{n+1} = x_n + f(x_n)\,\Delta t + g(x_n)\,\Delta W_n + \frac{1}{2}g(x_n)\,g'(x_n)\left[(\Delta W_n)^2 - \Delta t\right].
$$

The additional term $\frac{1}{2}g\,g'[(\Delta W)^2 - \Delta t]$ arises from a Taylor expansion of the diffusion term and raises the strong order of convergence to $1$.

For the stochastic Lorenz system with *additive* noise ($g = \sigma_n = \text{const}$, so $g' = 0$), the Milstein correction vanishes identically, and the method reduces to Euler–Maruyama. This is a fortunate simplification: it means that for our additive-noise model, there is no advantage to using Milstein over Euler–Maruyama. The distinction becomes important only if we adopt a multiplicative noise model, where $g$ depends on the state variables.

It is worth emphasizing that this equivalence is *specific to additive noise*. If one were to model, say, state-dependent thermal noise where the noise intensity scales with current (and hence with the state variable), the Milstein method would provide a genuine improvement in accuracy.

### 5.7.3 Higher-Order Stochastic Integrators

Beyond Euler–Maruyama and Milstein, there exist higher-order methods such as the stochastic Runge–Kutta schemes of Rößler and the strong-order 1.5 methods of Kloeden and Platen. These require generating correlated random increments (not just independent Gaussian samples) and are considerably more complex to implement. For the Lorenz system with moderate noise intensities, Euler–Maruyama with a sufficiently small time step (typically $\Delta t \leq 10^{-4}$) provides adequate accuracy, and we recommend it as the default choice for the computational notebook.

### 5.7.4 Convergence and Stability Considerations

A critical distinction in stochastic numerics is between *strong* and *weak* convergence. Strong convergence measures the pathwise error $\mathbb{E}[|\mathbf{x}(t) - \mathbf{x}_N|]$, which tracks how well the numerical solution approximates a specific trajectory for a given noise realization. Weak convergence measures the error in expectations $|\mathbb{E}[\phi(\mathbf{x}(t))] - \mathbb{E}[\phi(\mathbf{x}_N)]|$ for smooth test functions $\phi$, which tracks statistical properties.

For the synchronization analysis, weak convergence is usually sufficient: we care about the *mean-squared* error $\mathbb{E}[E]$ rather than the error on any single realization. This means Euler–Maruyama (weak order 1) is quite adequate.

Stability of the numerical scheme is also a concern. For the Lorenz system, the largest eigenvalue of the Jacobian can reach $|\lambda| \sim \sigma = 16$, imposing a stability limit of roughly $\Delta t < 2/\sigma \approx 0.125$ for explicit methods. In practice, one uses a time step much smaller than this (for example, $\Delta t = 10^{-3}$ to $10^{-4}$) to also resolve the fine structure of the attractor.

## 5.8 Comparison: Lorenz SCS vs. Extended Kalman Filter

Cuomo et al. draw an illuminating comparison between the Lorenz synchronizing chaotic system (SCS) and the extended Kalman filter (EKF) as competing approaches to state estimation in the presence of noise.

The Lorenz SCS is an *open-loop* nonlinear observer: the receiver equations use the drive signal $u(t)$ directly but do not adjust their gains based on the estimation error. This makes the SCS simple to implement (it is essentially a copy of the transmitter circuit with the drive signal fed in) but limits its ability to optimize noise rejection.

The EKF, by contrast, is a *closed-loop* estimator that linearizes the system about the current state estimate and uses the Riccati equation to compute an optimal gain $K(t)$ at each instant. The EKF minimizes the mean-squared estimation error under the assumption of Gaussian noise, making it theoretically optimal for small perturbations. However, the EKF requires accurate knowledge of the system parameters and the noise statistics, and it can diverge if the initial state estimate is poor, the chaotic sensitivity means that a bad initial guess leads to linearization about a wildly incorrect trajectory.

The SCS has the advantage of *global* convergence: synchronization occurs from any initial condition, as guaranteed by the Lyapunov analysis. The EKF has the advantage of better noise filtering above its convergence threshold. In practice, Cuomo et al. found that the two estimators perform comparably over a wide range of input SNRs, with the SCS being slightly more robust to initial-condition errors and the EKF being slightly better at high SNR.

## 5.9 Summary and Bridge to the Computational Notebook

This section has extended the deterministic Lorenz system to the stochastic domain, motivated by the physical reality of noise in electronic circuits and communication channels. The key theoretical results are:

1. The stochastic Lorenz system is an Itô SDE with additive noise, characterized by the noise intensity $\sigma_n$.

2. Noise blurs the strange attractor, destroys deterministic predictability, and replaces individual trajectories with probability densities governed by the Fokker–Planck equation.

3. Synchronization persists in the presence of noise, but the error converges to a nonzero steady-state mean-squared value $\mathbb{E}[E]_{\text{ss}} \propto \sigma_n^2 / \gamma$.

4. The output SNR for signal masking is $\text{SNR}_{\text{out}} \propto P_m \sigma \gamma / \sigma_n^2$, making explicit the trade-off between message power, synchronization strength, and noise intensity.

5. The Euler–Maruyama method is the appropriate numerical integrator for the additive-noise case, coinciding with the Milstein method since the diffusion coefficient is state-independent.

The computational notebook will implement these theoretical predictions numerically: simulating the stochastic Lorenz system, verifying attractor blurring, measuring synchronization error as a function of $\sigma_n$, and demonstrating message recovery under noisy conditions.

# Part 5, SDE Simulations


Computational companion to the Part 5 theoretical notebook. Adds runnable simulations of
the stochastic Lorenz system using the Euler-Maruyama method, with visualizations of how
noise affects the attractor, the synchronization quality, and the bit error rate of
binary chaos shift keying.

## Summary of the noise simulations

| Quantity | Behavior under noise |
|----------|---------------------|
| Attractor structure | Fine structure blurs as $\sigma_n$ increases. The two-lobed butterfly remains visible up to roughly $\sigma_n \approx 0.3$. |
| Synchronization error rms | Grows roughly linearly with $\sigma_n$. Quality is acceptable for $\sigma_n < 0.1$. |
| CSK bit error rate | Rises slowly at first, then degrades sharply once noise exceeds the parameter mismatch in the b-shift. The system becomes unusable as BER approaches 0.5. |

These results are consistent with the theoretical bounds in the Part 5 theory notebook: the
noise floor of the recovered message scales linearly with $\sigma_n$, and detection reliability
depends on the contrast between the matched and mismatched error powers.

For the physical circuit in Part 4, this gives a concrete noise-tolerance specification:
the analog noise floor of the implementation should stay below $\sigma_n \approx 0.05$ to
preserve high-quality message recovery and low bit error rate.

## 6. Conclusion

This notebook has developed a complete theoretical account of the Lorenz-based chaotic communication system introduced by Cuomo, Oppenheim, and Strogatz, progressing from the fluid-mechanical origins of the Lorenz equations through deterministic chaos and synchronization to the stochastic analysis required for any realistic implementation.

### 6.1 Summary of Key Results

The theoretical development yields several results that are central to understanding and evaluating the Cuomo system.

First, the Lorenz equations arise naturally from a well-defined physical system, thermally driven convection, and their chaotic behavior is not an artifact of ad hoc construction but a genuine consequence of the nonlinear interaction between convective transport and thermal diffusion. The three parameters $\sigma$, $r$, and $b$ have concrete physical interpretations as ratios of diffusivities, a normalized temperature gradient, and a geometric aspect ratio, respectively. The circuit-scaled form $(u, v, w)$ used by Cuomo et al. preserves the dynamical structure while mapping the state variables into the voltage range of practical operational amplifiers.

Second, the stability analysis reveals a precise bifurcation structure: the origin is stable for $r < 1$, a pair of symmetric fixed points $C^\pm$ are stable for $1 < r < r_{\text{crit}}$, and sustained chaos emerges for $r > r_{\text{crit}}$. The strange attractor that governs the chaotic regime is characterized by one positive and two negative Lyapunov exponents, confirming that the system is simultaneously expanding (producing sensitivity to initial conditions) and contracting (confining trajectories to a bounded, measure-zero set in state space).

Third, the Pecora–Carroll decomposition demonstrates that the Lorenz system admits a natural partition into drive and response subsystems, and the Lyapunov function $E(\mathbf{e}) = \frac{1}{2}(e_1^2/\sigma + e_2^2 + 4e_3^2)$ establishes that synchronization is globally asymptotically stable, the receiver locks onto the transmitter from any initial condition, with exponentially decaying error. This mathematical guarantee is what makes chaotic communication feasible: the receiver does not need to be initialized in any particular state.

Fourth, the stochastic extension shows that synchronization survives the addition of noise, but at a cost. The mean-squared synchronization error converges to a steady-state floor $\mathbb{E}[E]_{\text{ss}} = \sigma_n^2(1/\sigma + 5)/(2\gamma)$ that is proportional to the noise power and inversely proportional to the synchronization decay rate $\gamma$. The output signal-to-noise ratio for the masking scheme scales as $\text{SNR}_{\text{out}} \propto P_m \sigma \gamma / \sigma_n^2$, providing an explicit and testable relationship between system parameters and communication quality.

### 6.2 Limitations and Open Questions

Several limitations of the Cuomo approach are worth noting. The security of chaotic masking rests on the assumption that an eavesdropper cannot reconstruct the chaotic carrier from the transmitted signal $s(t) = u(t) + m(t)$. Subsequent work has shown that this assumption is fragile: if the eavesdropper knows or can estimate the system equations and parameters, techniques such as nonlinear forecasting, return-map analysis, and recursive parameter estimation can extract the message without a synchronized receiver. Modern assessments of chaotic communication therefore regard it as a form of *spread-spectrum modulation* rather than cryptographic encryption, it provides physical-layer security through signal obscurity, not computational hardness.

The additive noise model adopted in Section 5 is the simplest possible stochastic extension. Real circuits exhibit multiplicative noise (where the noise intensity depends on the signal level), colored noise (with a finite correlation time), and non-Gaussian perturbations (for example, shot noise in semiconductor junctions). A more complete analysis would incorporate these effects, potentially using the Stratonovich interpretation of the SDE to maintain consistency with the physical origin of multiplicative noise.

The Fokker–Planck equation for the three-dimensional Lorenz system does not admit closed-form solutions, and numerical solution on a three-dimensional grid is computationally expensive. Approximate methods, such as moment-closure techniques, path-integral representations, or Monte Carlo sampling, are required for quantitative predictions, and these are natural extensions of the simulation work in the computational notebook.

### 6.3 Connection to the Computational Notebook

The theoretical predictions developed here define a concrete agenda for numerical verification. The computational notebook will:

1. Simulate the deterministic Lorenz system using a fourth-order Runge–Kutta integrator, reproduce the strange attractor, and verify sensitivity to initial conditions by tracking the divergence of nearby trajectories.

2. Implement the transmitter–receiver synchronization scheme and confirm exponential decay of the error variables, reproducing the 45° synchronization plots (Figs. 5a and 5b of the Cuomo paper).

3. Simulate the stochastic Lorenz system using the Euler–Maruyama method, visualize attractor blurring as a function of $\sigma_n$, and measure the steady-state mean-squared error to test the theoretical bound $\mathbb{E}[E]_{\text{ss}} \propto \sigma_n^2$.

4. Demonstrate signal masking and binary modulation under noisy conditions, compute the output SNR, and compare to the theoretical prediction.

Together, the theory and computation notebooks provide a self-contained treatment of the Cuomo system, from the physics of convection to the statistics of noisy signal recovery.

### References

1. E. N. Lorenz, "Deterministic nonperiodic flow," *Journal of the Atmospheric Sciences*, vol. 20, pp. 130–141, 1963.

2. L. M. Pecora and T. L. Carroll, "Synchronization in chaotic systems," *Physical Review Letters*, vol. 64, pp. 821–824, 1990.

3. K. M. Cuomo, A. V. Oppenheim, and S. H. Strogatz, "Synchronization of Lorenz-based chaotic circuits with applications to communications," *IEEE Transactions on Circuits and Systems, II: Analog and Digital Signal Processing*, vol. 40, no. 10, pp. 626–633, 1993.

4. T. L. Carroll and L. M. Pecora, "Synchronizing chaotic circuits," *IEEE Transactions on Circuits and Systems*, vol. 38, pp. 453–456, 1991.

5. K. M. Cuomo and A. V. Oppenheim, "Synchronized chaotic circuits and systems for communications," MIT Research Laboratory of Electronics Technical Report 575, 1992.

6. S. H. Strogatz, *Nonlinear Dynamics and Chaos*, 2nd ed. Boulder, CO: Westview Press, 2015.

7. P. E. Kloeden and E. Platen, *Numerical Solution of Stochastic Differential Equations*. Berlin: Springer-Verlag, 1992.

8. B. Øksendal, *Stochastic Differential Equations: An Introduction with Applications*, 6th ed. Berlin: Springer-Verlag, 2003.

9. D. W. Jordan and P. Smith, *Nonlinear Ordinary Differential Equations*, 2nd ed. Oxford: Oxford University Press, 1987.

10. H. Risken, *The Fokker–Planck Equation: Methods of Solution and Applications*, 2nd ed. Berlin: Springer-Verlag, 1989.

---
*End of theoretical document.*